### Links and Resources
| Concept / Command | Description |
|-------------------|-------------|
| [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) | Canonical reference for sources, sinks, and semantics |
| [readStream API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.streaming.DataStreamReader.html) | The streaming counterpart to `spark.read` |
| [Auto Loader](https://docs.databricks.com/en/ingestion/cloud-object-storage/auto-loader/index.html) | Databricks' incremental file ingestion using `cloudFiles` |
| [Auto Loader options](https://docs.databricks.com/en/ingestion/cloud-object-storage/auto-loader/options.html) | Full reference of `cloudFiles.*` options |
| [Schema inference and evolution](https://docs.databricks.com/en/ingestion/cloud-object-storage/auto-loader/schema.html) | How Auto Loader handles changing schemas |
| [File notification mode](https://docs.databricks.com/en/ingestion/cloud-object-storage/auto-loader/file-notification-mode.html) | Scaling Auto Loader to billions of files |
| [rate source](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#input-sources) | Built-in test source emitting timestamped rows |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

DATA_ROOT = "/Volumes/population_metrics/landing/datasets"

# Streaming Sources and Auto Loader

Every streaming pipeline starts with a **source** — the place new rows come from. Pick the wrong source and nothing else matters: throughput, latency, and recovery semantics are all shaped by this choice.

This notebook covers:

1. The streaming sources Spark ships with.
2. The **rate source** — a built-in test generator we'll reuse in later notebooks.
3. The **file source** — read files as a stream, and why it's almost always the wrong choice on cloud storage.
4. **Auto Loader (`cloudFiles`)** — the Databricks-recommended way to ingest files, with schema inference and evolution built in.

## Streaming sources at a glance

| Source | Format string | Use case | Production-ready? |
|---|---|---|---|
| **Rate** | `rate` | Synthetic timestamped rows for tests and demos | No — testing only |
| **Rate per micro-batch** | `rate-micro-batch` | Same idea but emits a fixed number of rows per trigger | No — testing only |
| **File source** | `csv` / `json` / `parquet` / ... | Small directories with a fixed file pattern | Limited — see below |
| **Auto Loader** | `cloudFiles` | Files landing in cloud storage at any scale | Yes — the default on Databricks |
| **Delta Lake** | `delta` | Stream changes from a Delta table | Yes — covered in notebook 07 |
| **Kafka** | `kafka` | Real-time event streams from a Kafka cluster | Yes |
| **Socket** | `socket` | Read text lines from a TCP socket | No — debugging only |
| **Memory** | `memory` (sink, not source) | Hold output in driver memory for queries | No — testing only |

All sources share the same shape:

```python
df = (spark.readStream
        .format("<source>")
        .option("<option>", "<value>")
        .load(<path-or-nothing>))
```

What changes is the format and the options.

## The rate source — your testing workhorse

For finite-trigger demos we use **`rate-micro-batch`** rather than plain `rate`. It emits a fixed number of rows per micro-batch and plays nicely with `availableNow`:

| Option | Default | Meaning |
|---|---|---|
| `rowsPerBatch` | 1 | Rows emitted per trigger |
| `numPartitions` | (cluster default) | Parallelism — keep small for demos |

We'll write the stream to the **memory sink** with a query name and a finite trigger, await termination, and inspect what was produced.

> **Two serverless constraints to know.** This workspace uses `serverless_compute_id = auto`. (1) Serverless rejects the default `ProcessingTime` (continuous) trigger with `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`, so every `writeStream` here uses `.trigger(availableNow=True)`. (2) Serverless also blocks the implicit temp checkpoint dir that the memory sink would normally auto-create (`TEMP_CHECKPOINT_LOCATION_NOT_SUPPORTED`); you must pass `checkpointLocation` on every streaming write — we use a UUID-suffixed path so each run produces fresh output. On a classic cluster neither restriction applies.

In [ ]:
import time

"""
rate_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .option("numPartitions", 1)
    .load())
"""
rate_df = (spark.readStream
      .format("rate-micro-batch")
      .option("rowsPerBatch", 20)
      .option("numPartitions", 1)
      .load())

rate_df.printSchema()
print("isStreaming:", rate_df.isStreaming)

In [ ]:
import uuid

# Serverless workspace constraints on writeStream:
#   1. ProcessingTime trigger is rejected → use availableNow.
#   2. Implicit temp checkpoint locations are blocked → set checkpointLocation explicitly.
# Use a unique checkpoint path so each run is fresh — a persistent checkpoint would
# resume from the prior offset and emit zero new rows on the second run.
RATE_CHECKPOINT = f"{DATA_ROOT}/streaming_demo/_checkpoints/rate_demo_{uuid.uuid4().hex[:8]}"

query = (rate_df.writeStream
    .format("memory")
    .queryName("rate_demo")
    .outputMode("append")
    .option("checkpointLocation", RATE_CHECKPOINT)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()

display(spark.sql("SELECT * FROM rate_demo ORDER BY timestamp DESC LIMIT 10"))

Notice we got two columns for free:

- `timestamp` — wall-clock time the row was emitted (great for windowing examples).
- `value` — a monotonically increasing long.

We'll come back to this source in the windowing and joins notebooks because it's a reliable way to manufacture event-time data.

## The file source — and why Auto Loader replaces it

Reading files as a stream looks deceptively simple:

```python
(spark.readStream
   .format("json")                    # or csv, parquet, ...
   .schema(my_schema)                  # required for json/csv
   .option("maxFilesPerTrigger", 100)
   .load("/Volumes/.../landing/"))
```

It works, but the implementation **lists the entire source directory on every trigger** to find new files. That breaks down quickly:

| Problem | Why it hurts |
|---|---|
| Cloud listing is slow | `LIST` against S3/ADLS is paginated and rate-limited; minutes per trigger once you cross ~100K files |
| No schema inference | You must hand-write the schema; new columns are silently dropped |
| No schema evolution | Producer adds a column → consumer never sees it |
| Memory pressure | The driver holds the full file list between triggers |
| No retry of bad files | A malformed file fails the batch repeatedly |

On Databricks, **use Auto Loader instead**. The file source is fine for a one-off local demo or a directory with a handful of files.

## Auto Loader (`cloudFiles`)

Auto Loader is Databricks' production-grade file ingestion source. Same `readStream` shape, just `format("cloudFiles")` plus a few options.

Under the hood it solves every problem of the file source:

| Capability | What it does |
|---|---|
| **Incremental file discovery** | Tracks already-processed files in a RocksDB checkpoint — no re-listing the world every trigger |
| **Schema inference** | Samples files on first run and infers a schema |
| **Schema evolution** | When a new column appears, choose to fail, add it, rescue it, or ignore it |
| **Rescued data column** | Captures fields that didn't match the schema in a single `_rescued_data` column — no data loss |
| **Two discovery modes** | **Directory listing** (default, simple) and **file notification** (cloud event-driven, scales to billions of files) |
| **Format-specific options** | All the regular CSV/JSON/Parquet options still apply |

### The minimum viable Auto Loader query

```python
df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/.../_schemas/orders")
        .load("/Volumes/.../landing/orders/"))

(df.writeStream
   .option("checkpointLocation", "/Volumes/.../_checkpoints/orders")
   .trigger(availableNow=True)
   .toTable("population_metrics.landing.orders_clean"))
```

Three things you must supply:

1. **`cloudFiles.format`** — the underlying file format (`json`, `csv`, `parquet`, `avro`, `text`, `binaryFile`, ...).
2. **`cloudFiles.schemaLocation`** — directory where Auto Loader stores the inferred schema and tracks evolution.
3. **`checkpointLocation`** on the writer — durable location for offsets and commit log.

> Two separate locations on purpose: the schema location is read-only metadata about the source, the checkpoint is per-writer state. **Never share either across queries.**

### Live demo — drop a file, ingest it, drop another, evolve

We'll use a temporary location under the existing volume:

- **Source dir** — where we drop JSON files
- **Schema dir** — Auto Loader's inferred schema
- **Checkpoint dir** — the writer's progress
- **Output table** — where the parsed rows land

The trigger is `availableNow=True`: Auto Loader processes every currently-visible file, then stops. This is the canonical pattern for cost-efficient "streaming on a schedule".

In [ ]:
BASE = f"{DATA_ROOT}/streaming_demo"
SRC = f"{BASE}/orders_raw"
SCHEMA_DIR = f"{BASE}/_schemas/orders"
CHECKPOINT = f"{BASE}/_checkpoints/orders"
TABLE = "population_metrics.landing.orders_clean"

# Reset the demo paths and table so the notebook is re-runnable
dbutils = None
try:
    from databricks.sdk.runtime import dbutils  # works on cluster
except Exception:
    pass

spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
if dbutils is not None:
    for p in (SRC, SCHEMA_DIR, CHECKPOINT):
        try:
            dbutils.fs.rm(p, recurse=True)
        except Exception:
            pass
        dbutils.fs.mkdirs(p)

print("reset complete; SRC =", SRC)

In [ ]:
# Drop the first batch of JSON files into the landing directory
batch1 = [
    '{"order_id": 1, "customer": "Alice", "amount": 19.99}',
    '{"order_id": 2, "customer": "Bob",   "amount": 42.50}',
]

if dbutils is not None:
    for i, line in enumerate(batch1):
        dbutils.fs.put(f"{SRC}/orders_batch1_{i}.json", line, overwrite=True)

    display(spark.createDataFrame(dbutils.fs.ls(SRC)))

In [ ]:
# First run: Auto Loader infers schema, ingests files, commits, stops.
(spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("cloudFiles.schemaLocation", SCHEMA_DIR)
   .load(SRC)
   .writeStream
   .option("checkpointLocation", CHECKPOINT)
   .trigger(availableNow=True)
   .toTable(TABLE)
   .awaitTermination())

display(spark.table(TABLE))
spark.table(TABLE).printSchema()

### Schema evolution in action

Now imagine the producer adds a `priority` field. By default Auto Loader uses `addNewColumns` mode: it **fails the stream**, records the new schema, and on the next start picks up the new column.

The four modes:

| Mode | Behaviour when a new column appears |
|---|---|
| `addNewColumns` *(default)* | Fail the stream; the schema is updated so the next start includes the column |
| `failOnNewColumns` | Fail the stream and refuse to evolve until a human edits the schema |
| `rescue` | Succeed; new columns are captured in `_rescued_data` (a JSON string) |
| `none` | Succeed; new columns are silently ignored |

Set with `cloudFiles.schemaEvolutionMode`.

In [ ]:
# Drop a file with a new column 'priority' and a new column 'amount' as float still
batch2 = [
    '{"order_id": 3, "customer": "Carol", "amount": 7.25, "priority": "high"}',
]

if dbutils is not None:
    for i, line in enumerate(batch2):
        dbutils.fs.put(f"{SRC}/orders_batch2_{i}.json", line, overwrite=True)

In [ ]:
# Run again. With default mode the first attempt fails on the new column,
# the schema location is updated, and a second attempt succeeds.
from pyspark.errors import StreamingQueryException

def run_once():
    (spark.readStream
       .format("cloudFiles")
       .option("cloudFiles.format", "json")
       .option("cloudFiles.schemaLocation", SCHEMA_DIR)
       .load(SRC)
       .writeStream
       .option("checkpointLocation", CHECKPOINT)
       .option("mergeSchema", "true")          # let Delta accept the new column
       .trigger(availableNow=True)
       .toTable(TABLE)
       .awaitTermination())

try:
    run_once()                                  # may fail with NEW_FIELDS_IN_DATA
except Exception as e:
    print("first attempt failed (expected on schema change):", type(e).__name__)
    run_once()                                  # second attempt picks up the evolved schema

display(spark.table(TABLE).orderBy("order_id"))
spark.table(TABLE).printSchema()

### Schema hints and the rescued-data column

Two more options you'll reach for constantly:

**`cloudFiles.schemaHints`** — override inferred types without writing the full schema. Inference often picks `string` for numeric IDs or `double` for amounts; a hint nails it.

```python
.option("cloudFiles.schemaHints", "order_id long, amount decimal(10,2)")
```

**`_rescued_data`** — by default Auto Loader adds a column named `_rescued_data` (configurable via `cloudFiles.rescuedDataColumn`). Any field that couldn't be parsed into its inferred type — or any column that arrives in `rescue` evolution mode — is captured here as JSON. You lose nothing.

Inspect it after a run with: `SELECT _rescued_data FROM table WHERE _rescued_data IS NOT NULL`.

## Directory listing vs file notification mode

Auto Loader has two ways to discover new files:

| Mode | Set with | Trade-off |
|---|---|---|
| **Directory listing** *(default)* | `cloudFiles.useNotifications=false` | Zero setup. Lists the source directory each trigger. Optimised — uses sorted listing and incremental discovery — but ultimately bounded by `LIST` throughput. Good up to millions of files. |
| **File notification** | `cloudFiles.useNotifications=true` | One-time setup of a cloud event subscription (SNS/SQS, Event Grid, Pub/Sub). Files are pushed to a queue as they land — no listing at all. Required when you have billions of files or need very low latency. |

On AWS, file notification mode auto-creates the SNS topic and SQS queue. On Azure, the same for Event Grid + Storage Queue. You provide an IAM role / managed identity with the right permissions and Auto Loader handles the rest.

**Start with directory listing.** Switch to notification mode only when listing latency becomes a real bottleneck.

## Other sources, in one paragraph each

**Delta Lake** — `.format("delta").load(path)` or `.table("name")` reads a Delta table as a stream of appends. With `option("readChangeFeed", "true")` it streams **all** changes (inserts, updates, deletes) via Change Data Feed. Full coverage in notebook 07.

**Kafka** — `.format("kafka")` with `kafka.bootstrap.servers` and `subscribe`. Returns binary `key`/`value` columns plus partition/offset/timestamp metadata. Use `from_json` or `from_avro` to parse the payload. Mentioned here for completeness — needs a broker to run.

**Socket and console** — strictly for tutorials and ad-hoc debugging. Not durable. Don't put either in production.

## Cleanup

Streaming demos leave checkpoints, schema directories, and tables behind. Clean them up so re-running the notebook is deterministic.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

if dbutils is not None:
    for p in (SRC, SCHEMA_DIR, CHECKPOINT):
        try:
            dbutils.fs.rm(p, recurse=True)
        except Exception as e:
            print("skip", p, e)

print("cleanup done")

## Takeaways

- Use the **rate source** to prototype any streaming logic without external dependencies — we'll lean on it again in notebooks 04, 05, and 06.
- The plain **file source** is fine for tiny demos but breaks down on cloud storage. Reach for **Auto Loader** instead.
- An Auto Loader query needs **three durable locations**: source path, `cloudFiles.schemaLocation`, and writer `checkpointLocation`. Never reuse them across queries.
- `Trigger.AvailableNow` turns Auto Loader into a cheap, scheduled batch job that still gets all the streaming guarantees.
- Schema evolution defaults to `addNewColumns` — the stream fails once, then picks up the new column on restart. Combine with `mergeSchema` on the Delta sink for the end-to-end story.